In [0]:
%run ./helper

In [0]:
!ls 

In [0]:
#dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.dropdown("model_result_table", model_result_table_list[0], model_result_table_list)
dbutils.widgets.dropdown("instruct_model", instruct_model_list[0], instruct_model_list)
dbutils.widgets.dropdown("instruction_activity", instruction_activity_list[0], instruction_activity_list)
catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
model_result_table=dbutils.widgets.get("model_result_table")
instruct_model=dbutils.widgets.get("instruct_model")
instruction_activity=dbutils.widgets.get("instruction_activity")
df = spark.read.table(f"{catalog}.{schema}.{model_result_table}")
table_name = f"{catalog}.{schema}.{model_result_table}"
print(f"table_name:{table_name}, instruct_model:{instruct_model}, instruction_activity: {instruction_activity}" )


In [0]:
%run ./config/instruction_configuration

In [0]:
out_column_name_prefix=instructions[instruction_activity]["out_column_name_prefix"]
ai_query_logic=instructions[instruction_activity]["ai_query_logic"]

In [0]:
# 1. Configuration
model_id = instruct_model
column_name = f"{out_column_name_prefix}_{model_id.replace('databricks-', '').replace('-', '_').replace('_instruct', '').replace('.', '')}"

# 2. Add the column if it doesn't exist
# This avoids errors during the update phase
cols = [f.name for f in spark.table(table_name).schema.fields]

if column_name not in cols:
    print(f"Column {column_name} missing. Adding it now...")
    spark.sql(f"ALTER TABLE {table_name} ADD COLUMN {column_name} STRING")
else:
    print(f"Column {column_name} already exists. Skipping creation.")

# 4. Execute the Incremental Update
# Only updates rows where the column is currently NULL
merge_sql = f"""
MERGE INTO {table_name} AS target
USING (
  SELECT 
    model, 
    timestamp, 
    topic,
    {ai_query_logic} AS new_grade
  FROM {table_name}
  WHERE {column_name} IS NULL
) AS source
ON target.model = source.model 
   AND target.timestamp = source.timestamp 
   AND target.topic = source.topic
WHEN MATCHED THEN
  UPDATE SET target.{column_name} = source.new_grade
"""

print(f"Running incremental AI evaluation for: {column_name}...")
spark.sql(merge_sql)
print("Update successful.")

In [0]:
spark.table(f"{catalog}.{schema}.topic_info_local").select("topic_name_meta_llama_3_1_405b", "topic_name_gpt_5_2").display()